In [2]:

# IMPORTS
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.neighbors import KNeighborsClassifier


# LOAD DATA
df = pd.read_csv("data.csv")

# fix column spacing
df.columns = df.columns.str.strip()

# DROP COLUMNS
df = df.drop(columns=['Population', 'GDP'], errors='ignore')

# handle mixing values
# numeric
num_cols = df.select_dtypes(include=['int64','float64']).columns
for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df[col].fillna(df[col].median())

# categorical
cat_cols = df.select_dtypes(include=['str']).columns
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

# remove duplicates
df = df.drop_duplicates()

print("Cleaned Shape:", df.shape)

Cleaned Shape: (2938, 19)


In [7]:

# REGRESSION
# X and y
X_reg = df.drop(columns=['Life expectancy'])
y_reg = df['Life expectancy']

# encoding
X_reg = pd.get_dummies(X_reg)

# split
X_train, X_test, y_train, y_test = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

# scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


array([[-0.11655742, -0.31789671, -0.25751384, ..., -0.04397277,
        -0.46310937,  0.46310937],
       [-1.41779229, -0.10996316, -0.25751384, ..., -0.32060097,
        -0.46310937,  0.46310937],
       [ 0.75093249, -0.18194016, -0.25751384, ...,  0.2633919 ,
        -0.46310937,  0.46310937],
       ...,
       [-0.11655742,  0.8017455 , -0.1054732 , ..., -1.1197491 ,
        -0.46310937,  0.46310937],
       [-0.98404733, -0.74975871, -0.23962671, ...,  1.03180357,
         2.15931716, -2.15931716],
       [-0.55030238, -1.0536616 , -0.19490887, ..., -2.0418431 ,
        -0.46310937,  0.46310937]], shape=(2350, 19))

In [23]:

model1 = LinearRegression()
model1.fit(X_train, y_train)

pred1 = model1.predict(X_test)

mse1 = mean_squared_error(y_test, pred1)
rmse1 = np.sqrt(mse1)
r21 = r2_score(y_test, pred1)

print("=== Linear Regression ===")
print("MSE:", mse1)
print("RMSE:", rmse1)
print("R2:", r21)

=== Linear Regression ===
MSE: 15.363033695843573
RMSE: 3.9195706009515345
R2: 0.8227356555517287


In [24]:


model2 = DecisionTreeRegressor(random_state=42)
model2.fit(X_train, y_train)

pred2 = model2.predict(X_test)

mse2 = mean_squared_error(y_test, pred2)
rmse2 = np.sqrt(mse2)
r22 = r2_score(y_test, pred2)

print("\n=== Decision Tree ===")
print("MSE:", mse2)
print("RMSE:", rmse2)
print("R2:", r22)


=== Decision Tree ===
MSE: 6.59906462585034
RMSE: 2.568864462335516
R2: 0.9238575604250859


In [25]:
print("\n=== Comparison ===")

if r21 > r22:
    print("Linear Regression is better")
else:
    print("Decision Tree is better")


=== Comparison ===
Decision Tree is better


In [26]:

# CLASSIFICATION
# X and y
X_clf = df.drop(columns=['Status'])
y_clf = df['Status']

# encode target
y_clf = y_clf.map({'Developing':0, 'Developed':1})

# encode features
X_clf = pd.get_dummies(X_clf)

# split
X_train, X_test, y_train, y_test = train_test_split(
    X_clf, y_clf, train_size=0.7, random_state=42
)

# scaling that fit mean 0 and std 1 and this is x-mean/std 
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [27]:


clf1 = LogisticRegression(max_iter=1000)
clf1.fit(X_train, y_train)

pred1 = clf1.predict(X_test)

print("=== Logistic Regression ===")
print("Accuracy:", accuracy_score(y_test, pred1))
print("Confusion Matrix:\n", confusion_matrix(y_test, pred1))
print("Classification Report:\n", classification_report(y_test, pred1))

=== Logistic Regression ===
Accuracy: 0.9365079365079365
Confusion Matrix:
 [[704  33]
 [ 23 122]]
Classification Report:
               precision    recall  f1-score   support

           0       0.97      0.96      0.96       737
           1       0.79      0.84      0.81       145

    accuracy                           0.94       882
   macro avg       0.88      0.90      0.89       882
weighted avg       0.94      0.94      0.94       882



In [28]:

clf2 = KNeighborsClassifier(n_neighbors=5)
clf2.fit(X_train, y_train)

pred2 = clf2.predict(X_test)

print("\n=== KNN ===")
print("Accuracy:", accuracy_score(y_test, pred2))
print("Confusion Matrix:\n", confusion_matrix(y_test, pred2))
print("Classification Report:\n", classification_report(y_test, pred2))


=== KNN ===
Accuracy: 0.9648526077097506
Confusion Matrix:
 [[717  20]
 [ 11 134]]
Classification Report:
               precision    recall  f1-score   support

           0       0.98      0.97      0.98       737
           1       0.87      0.92      0.90       145

    accuracy                           0.96       882
   macro avg       0.93      0.95      0.94       882
weighted avg       0.97      0.96      0.97       882



In [29]:
acc1 = accuracy_score(y_test, pred1)
acc2 = accuracy_score(y_test, pred2)

print("\n=== Final Model Comparison ===")

if acc1 > acc2:
    print("Logistic Regression is better")
else:
    print("KNN is better")


=== Final Model Comparison ===
KNN is better
